# 02_feature_engineering

## Imports

In [20]:
from recruit_restaurant_visitor_forecasting.config import (
    CALENDAR_DATE_COL,
    GOLDEN_WEEK_FLG,
    AIR_RESTAURANT_ID_COL,
    VISIT_DATE_COL,
    OPEN_DATE_COL,
    AIR_AREA_COL,
    CITY_REGION_COL,
    VISITORS_COL,
    AIR_GENRE_COL,
    GENRE_TE,
    AREA_TE
)

In [21]:
from recruit_restaurant_visitor_forecasting.dataset import (
    DataDir,
    read_csv,
    save_csv,
)

In [22]:
from recruit_restaurant_visitor_forecasting.features import (
    add_holiday_columns,
    add_golden_week_flg,
    add_opened_recently_flg,
    get_first_str_values,
    add_open_flg,
    add_days_since_last_record,
    add_time_based_target_encoding
)

In [23]:
from recruit_restaurant_visitor_forecasting.dataset import (
    prepare_datetime_columns,
    standardize_date
)

In [24]:
air_visit_df = read_csv('air_visit.csv', DataDir.INTERIM)
air_reserve_df = read_csv('air_reserve.csv', DataDir.INTERIM)
hpg_reserve_df = read_csv('hpg_reserve.csv', DataDir.INTERIM)
sample_submission_df = read_csv('sample_submission.csv', DataDir.INTERIM)
date_info_df = read_csv('date_info.csv')
air_store_df = read_csv('air_store_info.csv')
hpg_store_df = read_csv('hpg_store_info.csv')
store_rel_df = read_csv('store_id_relation.csv')

In [25]:
prepare_datetime_columns(air_reserve_df)
prepare_datetime_columns(hpg_reserve_df)

standardize_date(date_info_df, CALENDAR_DATE_COL)
standardize_date(air_visit_df, VISIT_DATE_COL)
standardize_date(sample_submission_df, VISIT_DATE_COL)

## Date-dependent features

### Date info

It is necessary to add a feature for the distance to the nearest holiday.

In [26]:
date_info_df = add_holiday_columns(date_info_df, CALENDAR_DATE_COL)

It is also necessary to designate Golden Week, since not all days of this week are holidays.

In [27]:
date_info_df[GOLDEN_WEEK_FLG] = 0

for year in [2016, 2017]:
    add_golden_week_flg(date_info_df, year, CALENDAR_DATE_COL)

### Air visit

#### Opening dates

We can consider the restaurant's opening date. Since simply counting the number of days since opening may not be sufficient due to the increase in days over time, it's best to flag the restaurant's opening as occurring within the last six months. A **potential issue**: some restaurants either planned to open earlier than their minimum opening date but didn't, or didn't report visitors for earlier dates. This is indicated by the fact that air_reserve dataframe has reservations for earlier dates.

In [28]:
air_open_dates = air_visit_df.groupby(AIR_RESTAURANT_ID_COL)[VISIT_DATE_COL].min().rename(OPEN_DATE_COL)

In [29]:
air_visit_df = add_opened_recently_flg(air_visit_df, air_open_dates, VISIT_DATE_COL, AIR_RESTAURANT_ID_COL)
sample_submission_df = add_opened_recently_flg(sample_submission_df, air_open_dates, VISIT_DATE_COL, AIR_RESTAURANT_ID_COL)

In [30]:
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_within_last_six_months_flg
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1


In [31]:
sample_submission_df.head()

,id,visitors,air_store_id,visit_date,open_date,opened_within_last_six_months_flg
0,air_00a91d42b08b08d9_2017-04-23,0,air_00a91d42b08b08d9,2017-04-23,2016-07-01,0
1,air_00a91d42b08b08d9_2017-04-24,0,air_00a91d42b08b08d9,2017-04-24,2016-07-01,0
2,air_00a91d42b08b08d9_2017-04-25,0,air_00a91d42b08b08d9,2017-04-25,2016-07-01,0
3,air_00a91d42b08b08d9_2017-04-26,0,air_00a91d42b08b08d9,2017-04-26,2016-07-01,0
4,air_00a91d42b08b08d9_2017-04-27,0,air_00a91d42b08b08d9,2017-04-27,2016-07-01,0


#### Days since the last visit record

Let's say if a restaurant was open on the previous day and the current day, the value will be zero. Otherwise, it will be the number of days since the last opening plus one. An open day is defined as a day when the restaurant has more than zero customers (in the original dataset, there are always more than zero customers).

However, sample_submission has a problem: it doesn't have a concept of gaps in dates, meaning it's impossible to definitively determine when restaurants were open or closed. We must either rely on further calculation of the opening hours or remove this feature altogether.

In [32]:
air_visit_df = add_open_flg(air_visit_df, VISITORS_COL)

In [33]:
air_visit_df = add_days_since_last_record(air_visit_df, AIR_RESTAURANT_ID_COL, VISIT_DATE_COL)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_within_last_six_months_flg,is_open_flg,days_since_last_visit_record
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,1,0
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,1,0
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,0,1
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,1,2
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,1,0


#### City/region of area

For restaurants, it's best to separate the city and region (or, if necessary, just the city) without the district, since a relatively small number of restaurants belong to all three at once.

In [34]:
air_store_df[CITY_REGION_COL] = get_first_str_values(air_store_df[AIR_AREA_COL], 2)

#### Mean visitors by air city/region.

It is necessary to make smoothed target encoding, while the average value will be used for new areas. In this case, it is worth considering only days when the number of visitors is not zero, so that only open restaurants are taken into account.

In [35]:
air_visit_df = air_visit_df.merge(
    air_store_df,
    on=AIR_RESTAURANT_ID_COL,
    how="inner"
)

In [36]:
sample_submission_df = sample_submission_df.merge(
    air_store_df,
    on=AIR_RESTAURANT_ID_COL,
    how="inner"
)

In [39]:
air_visit_df, sample_submission_df = add_time_based_target_encoding(
    air_visit_df,
    sample_submission_df,
    CITY_REGION_COL,
    VISIT_DATE_COL,
    VISITORS_COL,
    AREA_TE
)

In [55]:
air_visit_df[[AIR_RESTAURANT_ID_COL, VISITORS_COL, AREA_TE, CITY_REGION_COL, VISIT_DATE_COL]].head()

,air_store_id,visitors,air_city_region_te,city_region,visit_date
0,air_00a91d42b08b08d9,35,26.239733,Tōkyō-to Chiyoda-ku,2016-07-01
1,air_00a91d42b08b08d9,9,26.382381,Tōkyō-to Chiyoda-ku,2016-07-02
2,air_00a91d42b08b08d9,0,26.289279,Tōkyō-to Chiyoda-ku,2016-07-03
3,air_00a91d42b08b08d9,20,26.232537,Tōkyō-to Chiyoda-ku,2016-07-04
4,air_00a91d42b08b08d9,25,26.049639,Tōkyō-to Chiyoda-ku,2016-07-05


In [56]:
sample_submission_df[[AIR_RESTAURANT_ID_COL, AREA_TE, CITY_REGION_COL, VISIT_DATE_COL]].head()

,air_store_id,air_city_region_te,city_region,visit_date
0,air_00a91d42b08b08d9,25.621639,Tōkyō-to Chiyoda-ku,2017-04-23
1,air_00a91d42b08b08d9,25.621639,Tōkyō-to Chiyoda-ku,2017-04-24
2,air_00a91d42b08b08d9,25.621639,Tōkyō-to Chiyoda-ku,2017-04-25
3,air_00a91d42b08b08d9,25.621639,Tōkyō-to Chiyoda-ku,2017-04-26
4,air_00a91d42b08b08d9,25.621639,Tōkyō-to Chiyoda-ku,2017-04-27


#### Mean visitors by air genre

It is necessary to make smoothed target encoding, while the average value will be used for new genres.

In [40]:
air_visit_df, sample_submission_df = add_time_based_target_encoding(
    air_visit_df,
    sample_submission_df,
    AIR_GENRE_COL,
    VISIT_DATE_COL,
    VISITORS_COL,
    GENRE_TE
)

In [53]:
air_visit_df[[AIR_RESTAURANT_ID_COL, VISITORS_COL, GENRE_TE, AIR_GENRE_COL, VISIT_DATE_COL]].head()

,air_store_id,visitors,air_genre_te,air_genre_name,visit_date
0,air_00a91d42b08b08d9,35,22.490114,Italian/French,2016-07-01
1,air_00a91d42b08b08d9,9,22.540151,Italian/French,2016-07-02
2,air_00a91d42b08b08d9,0,22.600815,Italian/French,2016-07-03
3,air_00a91d42b08b08d9,20,22.619089,Italian/French,2016-07-04
4,air_00a91d42b08b08d9,25,22.563322,Italian/French,2016-07-05


In [54]:
sample_submission_df[[AIR_RESTAURANT_ID_COL, GENRE_TE, AIR_GENRE_COL, VISIT_DATE_COL]].head()

,air_store_id,air_genre_te,air_genre_name,visit_date
0,air_00a91d42b08b08d9,22.565297,Italian/French,2017-04-23
1,air_00a91d42b08b08d9,22.565297,Italian/French,2017-04-24
2,air_00a91d42b08b08d9,22.565297,Italian/French,2017-04-25
3,air_00a91d42b08b08d9,22.565297,Italian/French,2017-04-26
4,air_00a91d42b08b08d9,22.565297,Italian/French,2017-04-27


### Restaurant's opening days

A one-hot encoding for the days of the week is needed. This data can be extracted based on the percentage of gaps (after processing, 0 visitors per day). In this case, there should most likely be no reservations for that day.

## Conclusion

### Added features.

- Days to/from the nearest holiday (negative is days until, positive is days after).
- Holiday indicator (currently included in date_info).
- Separate indicators for Golden Week dates.

- Indicator of whether the restaurant has been open within the last 6 months.
- Days since last recorded visit for air_visit dataframe.

- Smoothed target encoding of visitors by air_area_name.
- Smoothed target encoding of visitors by air_genre_name.

These features are currently in date_info dataframe, but will later be merged with air_visit.

Not exactly a feature, but I've put the city/region in a separate column for future features.
### Features planned for addition.

- Operating schedule - regular working days, extracted from the regular gaps in the data frame.

- Number of visitors on this day last month for this restaurant.
- Lag 1, 7, 28 of the number of visitors - for this restaurant/for neighbors.
- Missing lag indicator.
- Rolling mean/median/std of visitors over the past week, month - for this restaurant/for neighbors.

- Days since last recorded visit for sample_submission.

- Historical day-of-week mean/median up to (but not including) the current day - for this restaurant/for neighbors.

- Total reserved visitors (from air_reserve and hpg_reserve) on this day - for this restaurant/for neighbors.
- Rolling reserve/visitors difference over the past 7 / 28 days - for this restaurant/for neighbors.

### Possible features.

- Daily temperature - try to get the weather forecast.
- Precipitation probability.
- Hpg genre.

Decomposition features:
- Trend.
- Trend difference for last month.
- Seasonal.
- Residual mean for last month.